# 10 — Evaluation

**Purpose.** Federated versus centralised model: metrics with bootstrap confidence intervals, ROC curve and permutation importance.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/10_evaluation_*.png|pdf` and `reports/notebooks/10_evaluation.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "10_evaluation"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Train and evaluate

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.data_partitioning import partition
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig, centralized_baseline, build_model
from federated_learning.evaluation.metrics import evaluate
from federated_learning.training.models import set_flat
import torch
d = make_dataset(n_samples=6000, seed=SEED); parts = partition("dirichlet", d.y_train, 10, SEED, alpha=1.0)
fl = run_fl(FLConfig(rounds=20, seed=SEED), d, parts); cen = centralized_baseline(d, epochs=20)
print("federated:", {k: round(v, 3) for k, v in fl.final.items() if k in ("acc", "f1", "auc")}); print("centralised:", {k: round(v, 3) for k, v in cen.items() if k in ("acc", "f1", "auc")})

federated: {'acc': 0.854, 'f1': 0.798, 'auc': 0.939}
centralised: {'acc': 0.928, 'f1': 0.908, 'auc': 0.976}


## 2. Bootstrap 95% CI of the federated accuracy and ROC

In [3]:
from sklearn.metrics import roc_curve
m = fl.extras["model"]; set_flat(m, fl.w); m.eval()
with torch.no_grad(): p = torch.softmax(m(torch.from_numpy(d.X_test)), 1)[:, 1].numpy()
pred = (p >= .5).astype(int); rng = np.random.default_rng(SEED)
bs = [(pred[i] == d.y_test[i]).mean() for i in (rng.integers(0, len(p), len(p)) for _ in range(1000))]
ci = np.percentile(bs, [2.5, 97.5]); print("accuracy 95% CI:", ci.round(4)); SUMMARY["acc_ci"] = ci.tolist()
fpr, tpr, _ = roc_curve(d.y_test, p)

accuracy 95% CI: [0.8372 0.87  ]


## 3. Permutation importance of the global model and figure

In [4]:
from federated_learning.explainability import permutation_importance
imp = permutation_importance(build_model(40, 32, "relu"), fl.w, d.X_test, d.y_test, n_repeats=3, seed=SEED); top = np.argsort(imp)[::-1][:10]
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9)); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", color="grey"); ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate"); ax[0].set_title("ROC (federated model)")
ax[1].barh([f"f{j}" for j in top][::-1], imp[top][::-1]); ax[1].set_xlabel("accuracy drop"); ax[1].set_title("Permutation importance")
save_fig(fig, "evaluation"); SUMMARY["top_features"] = [int(j) for j in top]; save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/10_evaluation.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.